# 2.1 Travel Agent — HTTP MCP Server

## What you will learn in this notebook

- How to connect to a **hosted (remote) MCP server** over HTTP instead of stdio
- How a real-world **travel booking API** can be exposed as MCP tools
- How combining **MCP + memory** enables a conversational travel agent
- The difference between `stdio` and `streamable_http` transports

---

### stdio vs streamable_http

| | `stdio` transport | `streamable_http` transport |
|---|---|---|
| **Server location** | Your machine (subprocess) | Remote server (cloud/internet) |
| **How it starts** | LangChain spawns the process | Server is already running |
| **Connection** | stdin/stdout pipes | HTTP requests to a URL |
| **Latency** | Near-zero (local) | Network round-trip |
| **Use case** | Your own tools, `uvx` packages | Third-party SaaS APIs, hosted services |

### The Kiwi.com MCP Server

Kiwi.com is a flight search platform. They host an MCP server at `https://mcp.kiwi.com` that exposes their flight search API as MCP tools. We connect to it here — no API key required for basic usage.

This is the future of API integrations: instead of reading API docs, writing HTTP clients, and parsing response formats, you just point your agent at an MCP endpoint and the tools are ready to use.

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================
# CELL 2: Connect to the Kiwi.com MCP Server (HTTP Transport)
# ============================================================
# Key difference from 2.1_mcp.ipynb:
#   "transport": "streamable_http"  → connect over HTTP, not stdin/stdout
#   "url": "https://mcp.kiwi.com"  → the remote server address
#
# There is no "command" or "args" field because we don't launch
# a subprocess — the server is already running in Kiwi's cloud.
#
# LangChain sends MCP protocol messages as HTTP requests to this URL.
# The server responds with its tool definitions, and LangChain wraps
# them into standard tool objects — exactly the same interface we've
# used throughout this course, just sourced remotely.
#
# await is required because connecting and fetching tool schemas
# involves async network I/O.

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {                         # Nickname for this server
            "transport": "streamable_http",        # HTTP transport (not stdio)
            "url": "https://mcp.kiwi.com"          # Remote MCP server URL
        }
    }
)

# Fetch the flight search tools from Kiwi's server
tools = await client.get_tools()

In [ ]:
# ============================================================
# CELL 3: Create the Travel Agent
# ============================================================
# We combine three features for a useful travel assistant:
#
# tools=tools
#   → Kiwi.com flight search tools from the MCP server
#     The agent can search for real flights, prices, routes, etc.
#
# checkpointer=InMemorySaver()
#   → Multi-turn memory — user can refine the search without
#     repeating origin, destination, and date each time.
#     Follow-up: "Show me only direct flights" will work because
#     the agent remembers the original query.
#
# system_prompt="...No follow up questions."
#   → Key instruction: the agent should make a decision and return
#     results immediately rather than asking clarifying questions.
#     This creates a proactive assistant, not a passive one.
#     For a travel agent, users want results now — they can refine later.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,                    # Kiwi.com flight search tools
    checkpointer=InMemorySaver(),   # Remember the conversation
    system_prompt="You are a travel agent. No follow up questions."  # Proactive style
)

In [ ]:
# ============================================================
# CELL 4: Search for a Flight
# ============================================================
# The agent will:
#   1. Parse the user's request (SFO → TYO, March 31, direct only)
#   2. Call a Kiwi.com search tool with those parameters
#   3. Parse the flight results
#   4. Present the best options in natural language
#
# The word "direct" is key — the agent should filter or sort results
# to show only non-stop flights. Watch the tool_calls in the trace
# to see exactly what parameters it sends to Kiwi.
#
# Using ainvoke() because MCP tool calls are async (HTTP requests
# to the remote Kiwi server happen on each tool call).

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}  # Session for this user

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
)

In [ ]:
# ============================================================
# CELL 5: Inspect the Full Message Trace
# ============================================================
# For a flight search with one tool call, expect:
#   messages[0]  HumanMessage  → user's flight request
#   messages[1]  AIMessage     → tool call decision (flight search params)
#   messages[2]  ToolMessage   → raw flight results from Kiwi's API
#   messages[3]  AIMessage     → formatted flight options for the user
#
# The ToolMessage content (messages[2]) is worth inspecting —
# it shows the raw JSON from Kiwi's API, including prices, durations,
# airline codes, and booking links.

from pprint import pprint

pprint(response)

In [ ]:
# ============================================================
# CELL 6: Print Just the Final Answer
# ============================================================
# The agent's flight recommendations in natural language.
# Try adding a follow-up call using the same config:
#
#   response2 = await agent.ainvoke(
#       {"messages": [HumanMessage(content="What's the cheapest option?")]},
#       config  # Same thread_id — agent remembers the search results
#   )
#
# The memory means you can refine, compare, and drill into details
# without repeating the original destination and date.

print(response["messages"][-1].content)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **`streamable_http`** | Connect to remote MCP servers via URL — no subprocess, no local files |
| **Hosted MCP servers** | Third-party APIs (Kiwi, Stripe, GitHub, etc.) expose themselves as MCP endpoints |
| **`tools = await client.get_tools()`** | Same call regardless of transport type — stdio or HTTP, same interface |
| **"No follow up questions"** | System prompt trick to make agents proactive — return results immediately |
| **Memory + MCP** | `InMemorySaver` lets users refine searches without repeating all parameters |
| **`ainvoke()`** | Required when tools involve async I/O (HTTP calls to remote servers) |

### The bigger picture

This notebook shows why MCP matters for production apps. Instead of:
1. Reading Kiwi's API documentation
2. Writing an HTTP client with auth headers
3. Parsing their JSON response format
4. Writing a `@tool` wrapper

You just point `MultiServerMCPClient` at their URL and get working tools in two lines. This pattern scales: one MCP client can connect to flights, hotels, car rental, and weather simultaneously.